# AVDJ P1C3 — Correction commentée

Objectif : créer des features temporelles à partir du prix moyen au m² par ville.

On veut ajouter, pour chaque ville et chaque mois :

- le prix moyen au m² du mois précédent ;
- le nombre de transactions du mois précédent.

Ces variables servent ensuite à donner au modèle une information historique sans utiliser directement le futur.

In [1]:
from pathlib import Path

import pandas as pd
import polars as pl

## 1. Configuration

Le fichier attendu est `transactions_par_ville.parquet`.

Le code ci-dessous le cherche automatiquement dans le projet, y compris dans le dossier `data/`.

In [ ]:
PARQUET_FILENAME = "transactions_par_ville.parquet"

PRICE_M2_COLUMN = "prix_m2_moyen"
NB_TRANSACTIONS_PER_MONTH = "nb_transactions"

AGGREGATION_COLUMNS = [
    "departement",
    "ville",
    "id_ville",
]

SORT_COLUMNS = [
    "departement",
    "ville",
    "id_ville",
    "annee_transaction",
    "mois_transaction",
]


def find_parquet_file(filename: str = PARQUET_FILENAME) -> Path:
    """
    Cherche le fichier parquet dans le dossier courant et ses sous-dossiers.
    Cela évite de bloquer si le fichier est dans data/ ou dans un autre sous-dossier.
    """
    candidates = list(Path(".").rglob(filename))

    if not candidates:
        raise FileNotFoundError(
            f"Impossible de trouver {filename}. "
            "Place le fichier dans le projet ou dans le dossier data/."
        )

    return candidates[0]


parquet_path = find_parquet_file()
print(f"Fichier utilisé : {parquet_path}")

## 2. Version Polars

In [ ]:
transactions_per_city = pl.read_parquet(parquet_path)

# On affiche les colonnes pour vérifier que les noms attendus existent bien.
print("Colonnes disponibles :")
for column in transactions_per_city.columns:
    print("-", column)

print("
Aperçu :")
transactions_per_city.head()

In [ ]:
# Certaines colonnes peuvent être lues comme des chaînes de caractères.
# On les force en entiers pour que le tri chronologique soit correct.
transactions_per_city = transactions_per_city.with_columns(
    pl.col("departement").cast(pl.Int32),
    pl.col("mois_transaction").cast(pl.Int32),
    pl.col("annee_transaction").cast(pl.Int32),
)

In [ ]:
def compute_features_price_per_m2_polars(
    average_per_month_per_city: pl.DataFrame,
    aggregation_columns: list[str] = AGGREGATION_COLUMNS,
    sort_columns: list[str] = SORT_COLUMNS,
    price_m2_column: str = PRICE_M2_COLUMN,
    nb_transactions_column: str = NB_TRANSACTIONS_PER_MONTH,
) -> pl.DataFrame:
    """
    Ajoute des features du mois précédent avec Polars.

    Principe :
    - on trie les données par ville puis par date ;
    - pour chaque ville, on décale les valeurs d'une ligne avec shift(1) ;
    - ce décalage donne la valeur du mois précédent dans cette ville.

    Exemple :
    Janvier 2021 -> pas de mois précédent connu, donc null
    Février 2021 -> récupère les valeurs de janvier 2021
    Mars 2021    -> récupère les valeurs de février 2021
    """

    required_columns = set(
        aggregation_columns
        + sort_columns
        + [price_m2_column, nb_transactions_column]
    )
    missing_columns = required_columns - set(average_per_month_per_city.columns)

    if missing_columns:
        raise ValueError(
            "Colonnes manquantes dans le DataFrame : "
            + ", ".join(sorted(missing_columns))
        )

    result = (
        average_per_month_per_city
        .sort(sort_columns)
        .with_columns(
            # Prix moyen au m² du mois précédent pour la même ville
            pl.col(price_m2_column)
            .shift(1)
            .over(aggregation_columns)
            .alias("prix_m2_moyen_mois_precedent"),

            # Nombre de transactions du mois précédent pour la même ville
            pl.col(nb_transactions_column)
            .shift(1)
            .over(aggregation_columns)
            .alias("nb_transactions_mois_precedent"),
        )
        # La première ligne de chaque ville n'a pas de mois précédent.
        # On la retire donc pour éviter les valeurs nulles dans le modèle.
        .drop_nulls([
            "prix_m2_moyen_mois_precedent",
            "nb_transactions_mois_precedent",
        ])
    )

    return result

In [ ]:
transactions_with_features_polars = compute_features_price_per_m2_polars(
    transactions_per_city
)

print("Shape avant feature engineering :", transactions_per_city.shape)
print("Shape après feature engineering :", transactions_with_features_polars.shape)

transactions_with_features_polars.head()

## 3. Version Pandas

Même logique, mais avec `groupby(...).shift(1)`.

In [ ]:
transactions_per_city_pandas = pd.read_parquet(parquet_path)

transactions_per_city_pandas["departement"] = transactions_per_city_pandas["departement"].astype(int)
transactions_per_city_pandas["mois_transaction"] = transactions_per_city_pandas["mois_transaction"].astype(int)
transactions_per_city_pandas["annee_transaction"] = transactions_per_city_pandas["annee_transaction"].astype(int)

transactions_per_city_pandas.head()

In [ ]:
def compute_features_price_per_m2_pandas(
    average_per_month_per_city: pd.DataFrame,
    aggregation_columns: list[str] = AGGREGATION_COLUMNS,
    sort_columns: list[str] = SORT_COLUMNS,
    price_m2_column: str = PRICE_M2_COLUMN,
    nb_transactions_column: str = NB_TRANSACTIONS_PER_MONTH,
) -> pd.DataFrame:
    """
    Ajoute les mêmes features que la version Polars, mais avec Pandas.

    groupby(aggregation_columns) permet de faire le décalage ville par ville.
    shift(1) prend la valeur de la ligne précédente dans chaque groupe.
    """

    required_columns = set(
        aggregation_columns
        + sort_columns
        + [price_m2_column, nb_transactions_column]
    )
    missing_columns = required_columns - set(average_per_month_per_city.columns)

    if missing_columns:
        raise ValueError(
            "Colonnes manquantes dans le DataFrame : "
            + ", ".join(sorted(missing_columns))
        )

    result = average_per_month_per_city.copy()

    # Tri indispensable : sans tri chronologique, shift(1) pourrait prendre
    # une ligne précédente qui ne correspond pas au mois précédent.
    result = result.sort_values(sort_columns)

    result["prix_m2_moyen_mois_precedent"] = (
        result
        .groupby(aggregation_columns)[price_m2_column]
        .shift(1)
    )

    result["nb_transactions_mois_precedent"] = (
        result
        .groupby(aggregation_columns)[nb_transactions_column]
        .shift(1)
    )

    result = result.dropna(
        subset=[
            "prix_m2_moyen_mois_precedent",
            "nb_transactions_mois_precedent",
        ]
    )

    return result

In [ ]:
transactions_with_features_pandas = compute_features_price_per_m2_pandas(
    transactions_per_city_pandas
)

print("Shape avant feature engineering :", transactions_per_city_pandas.shape)
print("Shape après feature engineering :", transactions_with_features_pandas.shape)

transactions_with_features_pandas.head()

## 4. Pourquoi le scaling avant la séparation Train/Test est du data leakage ?

Le scaling calcule des statistiques sur les données, par exemple la moyenne et l'écart-type avec `StandardScaler`.

Si on applique le scaling **avant** la séparation train/test, le scaler apprend la moyenne et l'écart-type sur tout le dataset, y compris les données de test.

Cela crée une fuite d'information : le modèle d'entraînement bénéficie indirectement d'informations venant du jeu de test.

Conséquence : les performances mesurées peuvent être trop optimistes, car le test n'est plus totalement indépendant.

La bonne pratique est donc :

1. séparer les données en `X_train`, `X_test`, `y_train`, `y_test` ;
2. entraîner le scaler uniquement sur `X_train` avec `fit` ;
3. transformer `X_train` et `X_test` avec ce scaler ;
4. idéalement, utiliser un `Pipeline` Scikit-learn pour éviter les erreurs.

Exemple correct :

```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

model.fit(X_train, y_train)
score = model.score(X_test, y_test)
```